> **Archived original (v1.0, Colab, October 2026).** Kept for provenance; outputs stripped. It mounts Google Drive and reads files by absolute path, so it will not run as-is in this repository. The same pipeline now lives in `src/hvd/` and `scripts/run_index.py`; `python scripts/run_index.py --legacy` reproduces this notebook's scores and tiers exactly (see `tests/test_index.py`).

# Proposing an Empirical Case Study for Machine Learning Enabled Displacement Prediction and Environmental Justice

Extreme weather events, exacerbated by human-caused climate change, disproportionately impact communities with limited socioeconomic resources and political representation. For marginalized populations, a single extreme heatwave, drought, or flood event often triggers a cascade of internal displacement, economic instability, and physical insecurity. These populations frequently face violations of their rights under the Universal Declaration of Human Rights (UDHR), specifically Article 3 (the right to life and security) and Article 25 (the right to an adequate standard of living). Because these communities often live in regions with sparse ground-level monitoring, standard climate models effectively overlook them, leaving them vulnerable to unpredictable disasters. This project recognizes that climate-induced displacement is a profound societal problem that deepens existing inequalities and demands data-driven interventions to uphold human rights.

Environmental displacement presents a complex structural challenge at the intersection of climate change-induced extreme weather, forced human migration, and human rights. Focusing on internally displaced persons and marginalized populations, this research seeks to downscale climate data to create actionable insights for communities vulnerable to environmental shocks. Standard global circulation models inadvertently obscure localized extreme weather events, leading to high prediction uncertainty due to coarse geographic resolution. To address this predictive gap, this proposal outlines integrating physics-informed deep learning into the Extreme Weather and Environmental Displacement and Health initiative. By repurposing a Super Resolution Generative Adversarial Network to validate ERA5 reanalysis data against localized ground sensors, the Graduate Student Researcher bridges the gap between theoretical exascale climate modeling and practical humanitarian needs. This computational methodology operationalizes Restorative Justice and Distributive Justice by closing localized data gaps, enabling humanitarian agencies to transition from reactive aid distribution to proactive supply prepositioning. Ultimately, this work provides a concrete machine learning application for the convening's final strategic white paper and serves as the primary empirical case study for the researcher's dissertation chapter on global environmental justice and human rights advocacy.

## Libraries and Dependencies

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Plotting, Mapping, Animations
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # Added for the temporal slider animation
import folium # Added for interactive geospatial mapping

# File I/O
import glob # Added for file gathering
import os # Added for path joining
import requests
import json

# Show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Data Loading

In [ ]:
# Start with the INFORM Data
path_INFORM = "/content/drive/MyDrive/Documents/Fellowships/Convene_Influence_Grant/INFORM_Risk_Data"

#Load the data
clim_risk_mc = pd.read_csv(f"{path_INFORM}/IMF_CDIR_ECFRMC.csv")
clim_risk_mh = pd.read_csv(f"{path_INFORM}/IMF_CDIR_ECFRMH.csv")
clim_risk_mr = pd.read_csv(f"{path_INFORM}/IMF_CDIR_ECFRMR.csv")
clim_risk_mv = pd.read_csv(f"{path_INFORM}/IMF_CDIR_ECFRMV.csv")

In [ ]:
# Add the Climate Data
inform_risk_cc = pd.read_csv(f"{path_INFORM}/INFORM_CC.csv")
inform_risk_ha = pd.read_csv(f"{path_INFORM}/INFORM_HA.csv")
inform_risk_ovrl = pd.read_csv(f"{path_INFORM}/INFORM_OVRL.csv")
inform_risk_vu = pd.read_csv(f"{path_INFORM}/INFORM_VU.csv")

In [ ]:
inform_risk_vu.columns

In [ ]:
inform_risk_cc.head()

In [ ]:
inform_risk_cc.tail(20)

In [ ]:
inform_risk_ha.head(10)

In [ ]:
inform_risk_ovrl.head(10)

In [ ]:
inform_risk_vu.head(10)

In [ ]:
clim_risk_mc.head(10)

In [ ]:
clim_risk_mh.head(10)

In [ ]:
clim_risk_mr.head(10)

In [ ]:
clim_risk_mc['REF_AREA_LABEL'].unique()

In [ ]:
clim_risk_mv.head(10)

## Data Preprocessing, PCA, and t-SNE

To accommodate this dynamic complexity while ensuring direct operational utility, the computational framework will explicitly model its architecture on the Vulnerability Assessment Framework and the United Nations High Commissioner for Refugees International Detention Coalition (UNHCR-IDC) Vulnerability Screening Tool. Rather than an arbitrary statistical score, this approach evaluates distinct demographic sectors, including economic status, protection needs, food security, basic water and sanitation access, and underlying health conditions, alongside the physical climate data. The researcher will use dimensionality reduction techniques, specifically Principal Component Analysis and t-SNE, to consolidate these socioeconomic and environmental variables into a transparent, multifaceted index.

This approach ensures that social, livelihood, and economic vulnerabilities are accurately quantified and directly inform community resilience calculations. The consolidated data will be processed through agreed upon sectoral rules to cluster and rank communities into defined tiers of vulnerability: low, mild, moderate, and severe. This tiered categorization mirrors the scoring mechanisms humanitarian agencies use to prioritize targeted relief. By deploying spatial correlation techniques, the research will rigorously evaluate the complex relationships between each localized environmental anomaly, this tiered composite index, and the demographic population data. Furthermore, cluster analyses will identify and categorize specific displacement hotspots, operationally validating zones prioritized for proactive resource prepositioning and intervention within asylum and migration systems.

In [ ]:
def preprocess_raw_data(file_paths, output_dir=None):
    """
    Loads multiple raw CSV files, filters for the required columns,
    and merges the data into a wide format for PCA.
    """
    merged_df = None
    required_columns = ["REF_AREA", "REF_AREA_LABEL", "INDICATOR_LABEL", "TIME_PERIOD", "OBS_VALUE"]

    for file_path in file_paths:
        print(f"Processing {file_path}...")
        try:
            temp_df = pd.read_csv(file_path)

            # Ensure the columns exist before filtering to avoid KeyError
            missing_cols = [col for col in required_columns if col not in temp_df.columns]
            if missing_cols:
                print(f"Skipping {file_path}: Missing columns {missing_cols}")
                continue

            temp_df = temp_df[required_columns]

            # Sort by time period to keep the most recent observation for each area
            temp_df = temp_df.sort_values(by="TIME_PERIOD", ascending=False)
            temp_df = temp_df.drop_duplicates(subset=["REF_AREA", "REF_AREA_LABEL"], keep="first")

            # Rename OBS_VALUE to include the INDICATOR_LABEL and drop unnecessary columns
            indicator_name = str(temp_df["INDICATOR_LABEL"].iloc[0]).strip()
            temp_df = temp_df.rename(columns={"OBS_VALUE": f"{indicator_name}_VALUE"})
            temp_df = temp_df.drop(columns=["INDICATOR_LABEL", "TIME_PERIOD"])

            # Sequentially outer join each processed dataframe
            if merged_df is None:
                merged_df = temp_df
            else:
                merged_df = pd.merge(merged_df, temp_df, on=["REF_AREA", "REF_AREA_LABEL"], how="outer")

        except Exception as e:
            print(f"Warning: Could not process {file_path}. Error: {e}")

    if merged_df is None or merged_df.empty:
        raise ValueError("No data could be loaded. Check your file paths.")

    # Clean up for the PCA pipeline
    wide_df = merged_df.rename(columns={"REF_AREA_LABEL": "Region_ID"})
    wide_df = wide_df.drop(columns=["REF_AREA"])

    if output_dir:
        out_path = os.path.join(output_dir, "merged_raw_inform_data.csv")
        wide_df.to_csv(out_path, index=False)
        print(f"Saved intermediate merged raw data to {out_path}")

    return wide_df

def build_vulnerability_index(file_paths, output_dir=None):
    """
    Ingests World Bank INFORM indices, applies PCA for a composite score,
    and clusters into 4 UNHCR-aligned vulnerability tiers.
    """
    print("Loading and preprocessing raw World Bank index data...")
    df = preprocess_raw_data(file_paths, output_dir)

    # Step 1: Feature Selection & Preprocessing
    # Select only the numerical vulnerability/resilience columns
    features = df.drop(columns=['Region_ID'])

    # Handle any missing values from the World Bank dataset (using mean imputation)
    imputer = SimpleImputer(strategy='mean')
    imputed_features = imputer.fit_transform(features)

    # Standardize features so PCA isn't skewed by different scales
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(imputed_features)

    # Step 2: Principal Component Analysis (PCA)
    # We extract the first principal component to serve as our dynamic composite index
    print("Applying PCA to generate the composite vulnerability score...")
    pca = PCA(n_components=1)
    composite_score_raw = pca.fit_transform(scaled_features).flatten()

    # Normalize the PCA score to a 0-100 scale for humanitarian readability
    df['Vulnerability_Score_PCA'] = ((composite_score_raw - composite_score_raw.min()) /
                                     (composite_score_raw.max() - composite_score_raw.min())) * 100

    # Step 3: UNHCR-Aligned Clustering (Low, Mild, Moderate, Severe)
    print("Clustering regions into UNHCR operational tiers...")
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    # We cluster based on the PCA score to create distinct programmatic tiers
    clusters = kmeans.fit_predict(df[['Vulnerability_Score_PCA']])
    df['Cluster_ID'] = clusters

    # Map the arbitrary K-Means cluster IDs to logical UNHCR Tiers based on their centroids
    cluster_centers = kmeans.cluster_centers_.flatten()
    sorted_centers_idx = np.argsort(cluster_centers)

    # Mapping logic: Lowest centroid = Low, Highest = Severe
    tier_mapping = {
        sorted_centers_idx[0]: '1 - Low Vulnerability',
        sorted_centers_idx[1]: '2 - Mild Vulnerability',
        sorted_centers_idx[2]: '3 - Moderate Vulnerability',
        sorted_centers_idx[3]: '4 - Severe Vulnerability'
    }
    df['UNHCR_Tier'] = df['Cluster_ID'].map(tier_mapping)

    # Step 4: t-SNE for Dynamic Visualization
    print("Generating t-SNE coordinates for 2D geospatial plotting...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_results = tsne.fit_transform(scaled_features)
    df['tSNE_1'] = tsne_results[:, 0]
    df['tSNE_2'] = tsne_results[:, 1]

    print("\nProcessing Complete. Sample Output:")
    print(df[['Region_ID', 'Vulnerability_Score_PCA', 'UNHCR_Tier']].head())

    return df

def plot_tiers(df, output_dir="."):
    """
    Generates a quick scatter plot of the t-SNE results colored by UNHCR Tier
    to drop into your Wednesday presentation slides.
    """
    plt.figure(figsize=(10, 8))
    tier_order = ['1 - Low Vulnerability', '2 - Mild Vulnerability',
                  '3 - Moderate Vulnerability', '4 - Severe Vulnerability']

    sns.scatterplot(
        x='tSNE_1', y='tSNE_2',
        hue='UNHCR_Tier',
        hue_order=tier_order,
        palette=['#2ca02c', '#ff7f0e', '#d62728', '#9467bd'],
        data=df,
        legend="full",
        alpha=0.8,
        s=100
    )
    plt.title('t-SNE Projection of Regions by UNHCR Vulnerability Tier')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend(title='Operational Tier', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    plot_path = os.path.join(output_dir, 'unhcr_vulnerability_clusters.png')
    plt.savefig(plot_path, dpi=300)
    print(f"Saved cluster visualization to '{plot_path}'")
    plt.show()

def generate_vulnerability_map(df, output_dir="."):
    """
    Generates an interactive HTML map using Folium to display the PCA Vulnerability Score
    by directly modifying the underlying geographic polygon areas to guarantee alignment.
    """
    print("Generating interactive geospatial vulnerability map...")

    geojson_url = "https://raw.githubusercontent.com/python" + chr(45) + "visualization/folium/master/examples/data/world" + chr(45) + "countries.json"

    try:
        response = requests.get(geojson_url)
        geo_data = response.json()

        # Define the problem regions by their permanent polygon area ID and forcefully name them
        for feature in geo_data['features']:
            polygon_id = feature['id']

            # Target the Eritrea polygon area
            if polygon_id == 'ERI':
                feature['properties']['name'] = 'Eritrea'

            # Target the Palestinian Territory polygon area
            elif polygon_id == 'PSE':
                feature['properties']['name'] = 'West Bank and Gaza'

    except Exception as e:
        print(f"Failed to fetch or modify polygon areas: {e}")
        return

    # Initialize df_mapped BEFORE attempting to use it
    df_mapped = df.copy()

    # Safely duplicate specific rows to cover all potential GeoJSON naming conventions
    extra_rows = []

    if 'West Bank and Gaza' in df_mapped['Region_ID'].values:
        base_row = df_mapped[df_mapped['Region_ID'] == 'West Bank and Gaza'].iloc[0]
        for name in ['West Bank', 'Palestine', 'Gaza', 'Palestinian Territory']:
            new_row = base_row.copy()
            new_row['Region_ID'] = name
            extra_rows.append(new_row)

    if 'United States' in df_mapped['Region_ID'].values:
        base_row = df_mapped[df_mapped['Region_ID'] == 'United States'].iloc[0]
        new_row = base_row.copy()
        new_row['Region_ID'] = 'Puerto Rico'
        extra_rows.append(new_row)

    if 'Cyprus' in df_mapped['Region_ID'].values:
        base_row = df_mapped[df_mapped['Region_ID'] == 'Cyprus'].iloc[0]
        new_row = base_row.copy()
        new_row['Region_ID'] = 'Northern Cyprus'
        extra_rows.append(new_row)

    if len(extra_rows) > 0:
        df_mapped = pd.concat([df_mapped, pd.DataFrame(extra_rows)], ignore_index=True)

    # STEP 3: Dictionary to align the remaining World Bank nomenclature
    country_mapping = {
        'Viet Nam': 'Vietnam',
        'Venezuela, RB': 'Venezuela',
        'Bahamas, The': 'The Bahamas',
        "Cote d'Ivoire": 'Ivory Coast',
        'Congo, Rep.': 'Republic of the Congo',
        'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
        'Egypt, Arab Rep.': 'Egypt',
        'Yemen, Rep.': 'Yemen',
        'Iran, Islamic Rep.': 'Iran',
        'Turkiye': 'Turkey',
        'Syrian Arab Republic': 'Syria',
        'Czechia': 'Czech Republic',
        'Russian Federation': 'Russia',
        'Serbia': 'Republic of Serbia',
        'North Macedonia': 'Macedonia',
        'Kyrgyz Republic': 'Kyrgyzstan',
        'Lao PDR': 'Laos',
        'Korea, Rep.': 'South Korea',
        'Brunei Darussalam': 'Brunei',
        'Timor' + chr(45) + 'Leste': 'East Timor',
        'West Bank and Gaza': 'Palestine',
        'Gambia, The': 'Gambia',
        'Slovak Republic': 'Slovakia',
        'Micronesia, Fed. Sts.': 'Federated States of Micronesia',
        'Sao Tome and Principe': 'São Tomé and Principe',
        'United States': 'United States of America',
        'Tanzania': 'United Republic of Tanzania',
        'Eswatini': 'Swaziland',
        'Guinea' + chr(45) + 'Bissau': 'Guinea Bissau'
    }

    # Apply the dictionary mapping to the dataset
    df_mapped['Region_ID'] = df_mapped['Region_ID'].replace(country_mapping)

    # Strip any hidden trailing spaces that prevent exact string matching
    try:
      df_mapped['Region_ID'] = df_mapped['Region_ID'].astype(str).str.strip()
    except KeyError:
      print("Warning: Region_ID column not found. Ensure your PCA retains the country column.")

    esri_tiles = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
    esri_attr = "Tiles copyright Esri and Esri, DeLorme, NAVTEQ"

    # Initialize a global map centered on the equator using a clean Esri basemap
    vulnerability_map = folium.Map(
        location=[20, 0],
        zoom_start=2,
        tiles=esri_tiles,
        attr=esri_attr
    )

    # Add the choropleth layer to color regions by their PCA score
    # Use the locally modified geo_data dictionary instead of querying the URL again
    folium.Choropleth(
        geo_data=geo_data,
        name='Vulnerability Score',
        data=df_mapped,
        columns=['Region_ID', 'Vulnerability_Score_PCA'],
        key_on='feature.properties.name',
        fill_color='YlOrRd',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name='PCA Vulnerability Score (0 to 100)'
    ).add_to(vulnerability_map)

    # Save the interactive map
    map_path = os.path.join(output_dir, 'vulnerability_map.html')
    vulnerability_map.save(map_path)
    print("Saved interactive map to", map_path)

if __name__ == "__main__":
    path_INFORM = path_INFORM

    filenames = [
        "IMF_CDIR_ECFRMC.csv",
        "IMF_CDIR_ECFRMH.csv",
        "IMF_CDIR_ECFRMR.csv",
        "IMF_CDIR_ECFRMV.csv",
        "INFORM_CC.csv",
        "INFORM_HA.csv",
        "INFORM_OVRL.csv",
        "INFORM_VU.csv"
    ]

    dataset_paths = [os.path.join(path_INFORM, f) for f in filenames]

    try:
        processed_df = build_vulnerability_index(dataset_paths, output_dir=path_INFORM)
        plot_tiers(processed_df, output_dir=path_INFORM)

        # Generate the interactive Folium map
        generate_vulnerability_map(processed_df, output_dir=path_INFORM)

        # Export for your GIS/Mapping tool
        final_export = os.path.join(path_INFORM, "processed_vulnerability_tiers.csv")
        processed_df.to_csv(final_export, index=False)
        print(f"Saved final processed dataset to {final_export}")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        print("Please ensure your CSV files are correctly named and located in your working directory.")

## Measuring changes in vulnerability over time

In [ ]:
def generate_temporal_slider_map(df, output_dir="."):
    """
    Generates an interactive HTML map using Plotly with a temporal slider
    to visualize vulnerability shifts across historical periods.
    """
    print("Generating temporal geospatial map with slider...")

    # Ensure time period is an integer and sort the dataframe chronologically
    df_temporal = df.copy()

    # We use a try block in case the dataframe needs column renaming for the time period
    try:
        df_temporal['TIME_PERIOD'] = df_temporal['TIME_PERIOD'].astype(int)
        df_temporal = df_temporal.sort_values(by=['TIME_PERIOD'])

        # Create the animated choropleth projection
        fig = px.choropleth(
            df_temporal,
            locations="Region_ID",
            locationmode="country names",
            color="Vulnerability_Score_PCA",
            animation_frame="TIME_PERIOD",
            color_continuous_scale="YlOrRd",
            range_color=[0, 100],
            title="Longitudinal Vulnerability Trajectory"
        )

        # Save the animated map to a standalone HTML file
        out_path = os.path.join(output_dir, 'temporal_vulnerability_slider.html')
        fig.write_html(out_path)
        print("Saved temporal slider map to", out_path)

    except KeyError:
        print("Warning: TIME_PERIOD column not found. Ensure your PCA retains the year column.")

def execute_longitudinal_analysis(dataset_paths, output_dir):
    """
    Loads all time periods and fits a single global PCA to ensure
    parameters remain mathematically constant from year to year.
    """
    print("Executing longitudinal PCA pipeline...")
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    from sklearn.decomposition import PCA
    import os

    merged_df = pd.DataFrame()

    for file_path in dataset_paths:
        if not os.path.exists(file_path):
            print(f"Missing file: {file_path}")
            continue

        temp_df = pd.read_csv(file_path)
        indicator_name = temp_df['INDICATOR_LABEL'].iloc[0]

        temp_df = temp_df[['REF_AREA', 'REF_AREA_LABEL', 'TIME_PERIOD', 'OBS_VALUE']]
        temp_df = temp_df.rename(columns={'OBS_VALUE': indicator_name})

        if merged_df.empty:
            merged_df = temp_df
        else:
            merged_df = pd.merge(merged_df, temp_df, on=['REF_AREA', 'REF_AREA_LABEL', 'TIME_PERIOD'], how='outer')

    # Isolate numeric columns and impute missing data to prevent row dropping
    numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
    numeric_cols = numeric_cols.drop('TIME_PERIOD')
    merged_df[numeric_cols] = merged_df[numeric_cols].fillna(merged_df[numeric_cols].median())

    # Fit scaler and PCA on the entire longitudinal dataset simultaneously
    # This locks the eigenvectors so the parameters are identical for every year
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(merged_df[numeric_cols])

    pca = PCA(n_components=1, random_state=42)
    pca_scores = pca.fit_transform(scaled_data)

    # Scale scores from 0 to 100 for operational readability
    min_max = MinMaxScaler(feature_range=(0, 100))
    merged_df['Vulnerability_Score_PCA'] = min_max.fit_transform(pca_scores)

    # Ensure the region label matches what the Plotly function expects
    merged_df = merged_df.rename(columns={'REF_AREA_LABEL': 'Region_ID'})

    # Pass the chronological dataset to the plotting function
    generate_temporal_slider_map(merged_df, output_dir=output_dir)
    return merged_df

# The specific list of the eight files required for the PCA
filenames = [
    "IMF_CDIR_ECFRMC.csv",
    "IMF_CDIR_ECFRMH.csv",
    "IMF_CDIR_ECFRMR.csv",
    "IMF_CDIR_ECFRMV.csv",
    "INFORM_CC.csv",
    "INFORM_HA.csv",
    "INFORM_OVRL.csv",
    "INFORM_VU.csv"
]

# This list comprehension builds the exact dataset paths required by the function
dataset_paths = [os.path.join(path_INFORM, file_name) for file_name in filenames]

# Execute the temporal analysis and save the animated HTML map
execute_longitudinal_analysis(dataset_paths, output_dir=path_INFORM)

## Additional Geospatial Analyses

In [ ]:
def generate_flourish_style_rankings(df, output_dir=".", metric_col="Vulnerability_Score_PCA"):
    """
    Generates a Flourish style animated bar chart isolating the Top 10
    highest risk countries per year, with a scrubbing slider.
    """
    import plotly.express as px
    import pandas as pd
    import os

    print("Generating Flourish style interactive rankings...")

    # Safety check: if the column is missing, print what is actually available
    if metric_col not in df.columns:
        print(f"Error: Column '{metric_col}' not found.")
        print(f"Available columns are: {list(df.columns)}")
        return

    # 1. Composite Vulnerability Top 10 Bar Chart Race
    # Ensure we drop missing data for this specific metric and force the year to an integer
    risk_df = df.dropna(subset=[metric_col, 'TIME_PERIOD', 'Region_ID']).copy()
    risk_df['TIME_PERIOD'] = risk_df['TIME_PERIOD'].astype(int)

    # Loop through each year to isolate the top 10 countries dynamically
    top_10_list = []
    for year in sorted(risk_df['TIME_PERIOD'].unique()):
        year_df = risk_df[risk_df['TIME_PERIOD'] == year].copy()
        year_df = year_df.sort_values(metric_col, ascending=False).head(10)

        # Sort ascending so the highest value sits at the top of the Plotly horizontal bar chart
        year_df = year_df.sort_values(metric_col, ascending=True)
        top_10_list.append(year_df)

    animated_df = pd.concat(top_10_list)

    # Calculate the global minimum and maximum for the entire dataset
    global_min = risk_df[metric_col].min()
    global_max = risk_df[metric_col].max()

    # Generate the animated Plotly figure
    # You can update the title parameter here if you prefer a different custom title
    fig = px.bar(
        animated_df,
        x=metric_col,
        y='Region_ID',
        animation_frame='TIME_PERIOD',
        orientation='h',
        title='Countries at the Highest Overall Vulneability Risk',
        labels={'Region_ID': 'Country', metric_col: 'Composite Vulnerability Score'},
        range_x=[0, global_max * 1.05],
        color=metric_col,
        range_color=[global_min - 5, global_max],
        color_continuous_scale='OrRd'
    )

    # Slow down the animation timing for better readability and presentation pacing
    if "updatemenus" in fig.layout:
        fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 35000
        fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 5000

    # Enhance the layout for a clean, professional aesthetic
    fig.update_layout(
        plot_bgcolor='white',
        font_family="Inter, sans-serif",
        title_font_size=20,
        margin=dict(l=150, r=50, t=80, b=50)
    )

    out_path = os.path.join(output_dir, 'top_10_inform_risk.html')
    fig.write_html(out_path)
    print(f"Saved interactive ranking chart to {out_path}")

# Execute the longitudinal analysis and capture the dataframe
processed_longitudinal_df = execute_longitudinal_analysis(dataset_paths, output_dir=path_INFORM)

# Pass that dataframe into our new ranking function utilizing the PCA score
generate_flourish_style_rankings(processed_longitudinal_df, output_dir=path_INFORM)

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import os

def generate_comparative_metrics_dashboard(df, output_dir="."):
    """
    Generates an interactive Plotly horizontal bar chart with a dropdown menu.
    Allows the user to select specific underlying hazard and vulnerability metrics
    to view the top 10 highest risk countries for that specific category.
    """
    print("Generating comparative metrics dropdown visualization...")

    # Ensure TIME_PERIOD is numeric and filter for the most recent year available
    df = df.copy()
    df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
    latest_year = df['TIME_PERIOD'].max()
    recent_df = df[df['TIME_PERIOD'] == latest_year].copy()

    # Dynamically scan the dataframe to find all metric columns
    # This completely bypasses the need to guess the exact text strings
    available_metrics = {}
    for col in recent_df.columns:
        if col.endswith('_VALUE'):
            # Clean up the label for the dropdown menu
            display_label = col.replace('_VALUE', '').strip()
            available_metrics[display_label] = col

    # Fallback mechanism in case the suffix is missing
    if not available_metrics:
        exclude = ['TIME_PERIOD', 'Cluster_ID', 'tSNE_1', 'tSNE_2', 'Vulnerability_Score_PCA']
        numeric_cols = recent_df.select_dtypes(include=['number', 'float64', 'int64']).columns
        for col in numeric_cols:
            if col not in exclude:
                available_metrics[col] = col

    fig = go.Figure()
    dropdown_buttons = []

    if not available_metrics:
        print("Error: Could not identify metric columns.")
        print(f"Available columns: {list(recent_df.columns)}")
        return

    for i, (metric_name, col_name) in enumerate(available_metrics.items()):

        # Isolate the top 10 for this specific metric
        metric_df = recent_df[['Region_ID', col_name]].dropna()
        top_10 = metric_df.sort_values(by=col_name, ascending=False).head(10)

        # Sort ascending so the highest value sits at the top of the horizontal bar chart
        top_10 = top_10.sort_values(by=col_name, ascending=True)

        # Calculate the global minimum for this specific metric to properly scale the color gradient
        global_min = metric_df[col_name].min()
        global_max = metric_df[col_name].max()

        # Add the bar trace to the figure
        fig.add_trace(
            go.Bar(
                x=top_10[col_name],
                y=top_10['Region_ID'],
                orientation='h',
                name=metric_name,
                marker=dict(
                    color=top_10[col_name],
                    colorscale='OrRd',
                    cmin=global_min - 0.5, # Baseline slightly below minimum for intensity
                    cmax=global_max
                ),
                text=top_10[col_name].round(2),
                textposition='inside',
                visible=(i == 0) # Only the very first trace is visible upon loading
            )
        )

        # Create the visibility boolean array for the dropdown button
        # This tells Plotly which trace to show and which ones to hide when the button is clicked
        visibility = [False] * len(available_metrics)
        visibility[i] = True

        button = dict(
            label=metric_name,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Top 10 Displacement Hotspots: {metric_name} ({int(latest_year)})",
                 "xaxis": {"title": "Index Score"}}
            ]
        )
        dropdown_buttons.append(button)

    first_metric_name = list(available_metrics.keys())[0]

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=dropdown_buttons,
            x=0.0,
            xanchor="left",
            y=1.15,
            yanchor="top",
            direction="down",
            showactive=True,
            bgcolor="white",
            bordercolor="#cbd5e1",
            borderwidth=1
        )],
        title=f"Top 10 Displacement Hotspots: {first_metric_name} ({int(latest_year)})",
        title_font_size=20,
        plot_bgcolor='white',
        font_family="Inter, sans-serif",
        margin=dict(l=150, r=50, t=130, b=50), # Increased top margin to make room for the dropdown
        xaxis=dict(title="Index Score", showgrid=True, gridcolor='#e2e8f0'),
        yaxis=dict(title="")
    )

    # Save the interactive HTML file
    out_path = os.path.join(output_dir, 'comparative_risk_metrics.html')
    fig.write_html(out_path)
    print(f"Saved interactive dropdown chart to {out_path}")

# Execute the function using your previously processed longitudinal dataframe
# Ensure 'processed_longitudinal_df' and 'path_INFORM' are already defined in your notebook memory
generate_comparative_metrics_dashboard(processed_longitudinal_df, output_dir=path_INFORM)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def perform_pca_analysis(df, feature_cols):
    """
    Performs PCA on the features and prepares scree plot and feature loading data.
    """
    print("--- Starting PCA Analysis ---")
    imputed_data = df[feature_cols].fillna(df[feature_cols].median())

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(imputed_data)

    n_comp = min(3, len(feature_cols))
    pca = PCA(n_components=n_comp, random_state=42)
    pca.fit(scaled_data)

    individual_variance = [round(float(v) * 100, 2) for v in pca.explained_variance_ratio_]
    cumulative_variance = [round(float(v) * 100, 2) for v in np.cumsum(pca.explained_variance_ratio_)]
    components_labels = [f"PC{i+1}" for i in range(n_comp)]

    variance_data = {
        'components': components_labels,
        'individual_variance': individual_variance,
        'cumulative_variance': cumulative_variance
    }

    loadings = pca.components_
    z_data = []
    for i in range(len(feature_cols)):
        row_loadings = [round(float(loadings[pc, i]), 3) for pc in range(n_comp)]
        z_data.append(row_loadings)

    loadings_data = {
        'features': list(feature_cols),
        'pc_labels': components_labels,
        'z_matrix': z_data
    }

    return scaled_data, pca, variance_data, loadings_data

def perform_kmeans_validation(scaled_data, max_k=8):
    """
    Calculates WCSS and silhouette scores for different k values.
    """
    print("--- Validating K-Means Clusters ---")
    k_values = list(range(2, min(max_k + 1, len(scaled_data))))
    wcss = []
    silhouette_scores = []

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(scaled_data)
        wcss.append(round(float(kmeans.inertia_), 3))
        score = silhouette_score(scaled_data, clusters)
        silhouette_scores.append(round(float(score), 3))

    kmeans_data = {
        'k_values': k_values,
        'wcss': wcss,
        'silhouette_scores': silhouette_scores
    }
    return kmeans_data

def perform_tsne_analysis(scaled_data, n_components=2):
    """
    Reduces high-dimensional data into 2D space for cluster visualization.
    """
    print("\n--- Starting t-SNE Dimensionality Reduction ---")
    tsne = TSNE(n_components=n_components, random_state=42, perplexity=min(30, len(scaled_data) - 1))
    tsne_results = tsne.fit_transform(scaled_data)

    tsne_data = {
        'x': [round(float(val), 3) for val in tsne_results[:, 0]],
        'y': [round(float(val), 3) for val in tsne_results[:, 1]]
    }
    return tsne_data

def analyze_feature_importance(df, feature_cols, scaled_data, pca_model):
    """
    Calculates relative feature importances using Random Forest Regression on PC1.
    """
    print("\n--- Analyzing Feature Importance (Drivers of Vulnerability) ---")
    target_score = pca_model.transform(scaled_data)[:, 0]
    X = df[feature_cols].fillna(df[feature_cols].median())

    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, target_score)

    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]

    importance_data = {
        'features': [feature_cols[i] for i in indices],
        'importances': [round(float(importances[i]), 3) for i in indices]
    }
    return importance_data

def plot_pca_results(variance_data, loadings_data):
    print("--- Rendering PCA Plots ---")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].bar(variance_data['components'], variance_data['individual_variance'], alpha=0.7, label='Individual Variance')
    axes[0].plot(variance_data['components'], variance_data['cumulative_variance'], marker='o', color='r', label='Cumulative Variance')
    axes[0].set_title('PCA Scree Plot: Variance Explained')
    axes[0].set_ylabel('Variance Explained (%)')
    axes[0].legend()

    sns.heatmap(np.array(loadings_data['z_matrix']).T,
                xticklabels=loadings_data['features'],
                yticklabels=loadings_data['pc_labels'],
                cmap='coolwarm', center=0, ax=axes[1], annot=True, fmt=".2f")
    axes[1].set_title('PCA Feature Loadings (Correlation with Components)')

    plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

def plot_kmeans_validation(kmeans_data):
    print("--- Rendering K-Means Validation Plots ---")
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(kmeans_data['k_values'], kmeans_data['wcss'], marker='o', linestyle='-', color='b')
    axes[0].set_title('Elbow Method (Within-Cluster Sum of Squares)')
    axes[0].set_xlabel('Number of Clusters (k)')
    axes[0].set_ylabel('WCSS')

    axes[1].plot(kmeans_data['k_values'], kmeans_data['silhouette_scores'], marker='s', linestyle='-', color='g')
    axes[1].set_title('Silhouette Score Analysis')
    axes[1].set_xlabel('Number of Clusters (k)')
    axes[1].set_ylabel('Silhouette Score')

    plt.tight_layout()
    plt.show()

def plot_tsne(tsne_data):
    print("--- Rendering t-SNE Plot ---")
    plt.figure(figsize=(10, 7))
    plt.scatter(tsne_data['x'], tsne_data['y'], alpha=0.7, edgecolors='w', s=60, c='purple')
    plt.title('t-SNE Dimensionality Reduction (Cluster Visualization)')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.tight_layout()
    plt.show()

def plot_feature_importance(importance_data):
    print("--- Rendering Feature Drivers Plot ---")
    plt.figure(figsize=(12, 6))
    sns.barplot(x=importance_data['importances'], y=importance_data['features'], palette='viridis')
    plt.title('Feature Drivers of Vulnerability (Random Forest Regressor on PC1)')
    plt.xlabel('Relative Feature Importance')
    plt.ylabel('Vulnerability Indicators')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    if 'processed_longitudinal_df' in globals():
        df_vulnerability = processed_longitudinal_df.copy()
    else:
        print("Notice: 'processed_longitudinal_df' not found. Generating dummy dataset for testing.")
        np.random.seed(42)
        dummy_data = {
            'Region_ID': range(1, 301),
            'Temperature_Anomaly': np.random.normal(1.2, 0.8, 300),
            'Drought_Frequency': np.random.poisson(2, 300),
            'Poverty_Rate': np.random.uniform(15, 65, 300),
            'Healthcare_Access': np.random.uniform(20, 95, 300)
        }
        df_vulnerability = pd.DataFrame(dummy_data)

    exclude = ['TIME_PERIOD', 'Cluster_ID', 'tSNE_1', 'tSNE_2', 'Vulnerability_Score_PCA']
    feature_columns = [col for col in df_vulnerability.select_dtypes(include=[np.number]).columns
                       if col not in exclude and col != 'Region_ID']

    print(f"Using dataset with shape: {df_vulnerability.shape}")
    print(f"Detected Features: {feature_columns}\n")

    scaled_matrix, pca_model, pca_variance, pca_loadings = perform_pca_analysis(df_vulnerability, feature_columns)
    kmeans_validation = perform_kmeans_validation(scaled_matrix, max_k=8)
    tsne_results = perform_tsne_analysis(scaled_matrix)
    feature_drivers = analyze_feature_importance(df_vulnerability, feature_columns, scaled_matrix, pca_model)

    print("\n========================================================")
    print("DISPLAYING STATISTICAL VISUALIZATIONS")
    print("========================================================\n")

    plot_pca_results(pca_variance, pca_loadings)
    plot_kmeans_validation(kmeans_validation)
    plot_tsne(tsne_results)
    plot_feature_importance(feature_drivers)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

sns.set_theme(style="whitegrid")

def calculate_descriptive_stats(df, feature_cols):
    print("\n--- Descriptive Statistics ---")
    stats_df = df[feature_cols].describe().T

    # Adding measures of shape to check for normal distributions
    stats_df['skewness'] = df[feature_cols].skew()
    stats_df['kurtosis'] = df[feature_cols].kurtosis()

    print(stats_df[['mean', 'std', 'min', 'max', 'skewness', 'kurtosis']].round(3))
    return stats_df

def plot_correlation_matrix(df, feature_cols):
    print("\n--- Pearson Correlation Matrix ---")
    corr = df[feature_cols].corr(method='pearson')

    # Generate a mask for the upper triangle to remove redundancy
    mask = np.triu(np.ones_like(corr, dtype=bool))

    plt.figure(figsize=(10, 8))

    # Render the heatmap using the YlOrRd palette
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
                cmap='YlOrRd', vmin=-1, vmax=1,
                square=True, linewidths=.5, cbar_kws={"shrink": .75})

    plt.title('Pearson Correlation Matrix', fontsize=16, pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return corr

def perform_ttest(df, target_col, split_col, threshold=None):
    print(f"\n--- Two-Sample T-Test: {target_col} split by {split_col} ---")

    # If no threshold is provided, split the data by the median
    if threshold is None:
        threshold = df[split_col].median()

    group1 = df[df[split_col] >= threshold][target_col]
    group2 = df[df[split_col] < threshold][target_col]

    # Perform Welch's t-test (assumes unequal population variances)
    t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)

    print(f"Threshold for {split_col}: {threshold:.2f}")
    print(f"Group 1 (>= threshold) Mean {target_col}: {group1.mean():.2f} (n={len(group1)})")
    print(f"Group 2 (< threshold) Mean {target_col}: {group2.mean():.2f} (n={len(group2)})")
    print(f"T-Statistic: {t_stat:.4f}, P-Value: {p_val:.4e}")

    if p_val < 0.05:
        print("Conclusion: The difference in means is statistically significant (p < 0.05).")
    else:
        print("Conclusion: No statistically significant difference in means.")

    return t_stat, p_val

def perform_ols_regression(df, dependent_var, independent_vars):
    print(f"\n--- OLS Multiple Regression ---")
    print(f"Dependent Variable: {dependent_var}")
    print(f"Predictors: {', '.join(independent_vars)}")

    X = df[independent_vars]
    # Add a constant (intercept) to the model
    X = sm.add_constant(X)
    y = df[dependent_var]

    # Fit the Ordinary Least Squares model
    model = sm.OLS(y, X).fit()
    print(model.summary())

    return model

if __name__ == "__main__":
    # Fallback dataset generation to ensure script is fully runnable standalone
    if 'processed_longitudinal_df' not in globals():
        print("Notice: 'processed_longitudinal_df' not found in globals.")
        print("Generating a representative dataset to ensure standalone execution...\n")
        np.random.seed(42)
        dummy_data = {
            'Region_ID': range(1, 301),
            'Temperature_Anomaly': np.random.normal(1.2, 0.8, 300),
            'Drought_Frequency': np.random.poisson(2, 300),
            'Poverty_Rate': np.random.uniform(15, 65, 300),
            'Healthcare_Access': np.random.uniform(20, 95, 300),
            'Infrastructure_Quality': np.random.uniform(1, 10, 300),
            'Sea_Level_Rise': np.random.exponential(0.5, 300)
        }

        # Introduce some structural correlation so our statistical tests pick up realistic signals
        dummy_data['Poverty_Rate'] += dummy_data['Drought_Frequency'] * 3.5
        dummy_data['Poverty_Rate'] -= dummy_data['Infrastructure_Quality'] * 2.0
        dummy_data['Poverty_Rate'] = np.clip(dummy_data['Poverty_Rate'], 0, 100)

        processed_longitudinal_df = pd.DataFrame(dummy_data)

    df_vulnerability = processed_longitudinal_df.copy()

    # Identify numeric columns for the statistical analysis
    exclude = ['TIME_PERIOD', 'Cluster_ID', 'Region_ID']
    feature_columns = [col for col in df_vulnerability.select_dtypes(include=[np.number]).columns
                       if col not in exclude]

    print(f"Using historical database with shape: {df_vulnerability.shape}")
    print(f"Detected Features: {feature_columns}\n")

    # 1. Evaluate central tendency and distribution properties
    desc_stats = calculate_descriptive_stats(df_vulnerability, feature_columns)

    # 2. Visualize collinearity with a lower-triangle heatmap
    corr_matrix = plot_correlation_matrix(df_vulnerability, feature_columns)

    # 3. Choose columns dynamically depending on dummy or real dataset
    if 'Drought_Frequency' in df_vulnerability.columns and 'Poverty_Rate' in df_vulnerability.columns:
        target = 'Poverty_Rate'
        split_by = 'Drought_Frequency'
    else:
        target = 'Vulnerability_Score_PCA'
        split_by = 'Climate-Driven Hazard & Exposure'

    # Test hypothesis: Does exposure score split lead to worse overall vulnerability score?
    perform_ttest(df_vulnerability, target_col=target, split_col=split_by)

    # 4. Model the drivers of a dependent variable using OLS Regression
    independent_features = [f for f in feature_columns if f != target]
    perform_ols_regression(df_vulnerability, dependent_var=target, independent_vars=independent_features)

#Statistical Analysis Summary
Based on the descriptive and inferential statistics generated, here is a breakdown of our current dataset's behavior and the relationships between our risk indicators.
## 1. Descriptive Statistics & Data Health
The descriptive statistics indicate that our dataset is normally distributed. The skewness and kurtosis values across all indicators are close to $0$, meaning we don't have extreme outliers or heavily skewed tails that would require complex transformations (like log transformations) before running further statistical tests. The vulnerability indicators generally sit on a scale with means around $3.5$ to $4.5$.

## 2. T-Test: Hazard Impact on Vulnerability
The two-sample T-test provided an overwhelmingly clear result. By splitting the dataset into "High" and "Low" climate-driven hazard areas (using a threshold of $3.45$), we proved that regions with higher hazards have a drastically higher average PCA Vulnerability Score ($49.38$ vs $35.10$). With a p-value of $1.29 \times 10^{-97}$, this difference is statistically highly significant ($p < 0.01$), mathematically validating the core premise that hazard exposure acts as a primary stressor on overall vulnerability.

## 3. OLS Regression & Data Leakage
(The $R^2 = 1.0$ Issue)The OLS regression returned an $R^2$ of $1.000$, standard errors in the range of $10^{-15}$, and T-statistics in the quadrillions ($10^{15}$).In traditional statistics, an $R^2$ of $1.000$ is a major red flag indicating perfect collinearity or data leakage. Because the dependent variable (`Vulnerability_Score_PCA`) was mathematically constructed directly from the independent variables (the INFORM indices) using Principal Component Analysis, running a linear regression on it essentially just reverse-engineers the PCA equation. The model perfectly predicts the score because the score is the variables.

Additionally, looking at the features list, we have near-duplicate columns (e.g., `Index for Risk Management; Lack of Coping Capacity` and `Index for Risk Management: Lack of coping capacity index`). This redundancy will cause severe multicollinearity in future predictive modeling.

# Strategy for the Next Round of Statistics
To move past tautological (self-fulfilling) models, our next round of statistics should focus on the following:

  ### 1. Data Cleaning & Feature Selection
  We need to immediately identify and drop duplicate columns (the ones differing only by a semicolon vs. a colon). We should isolate the strict baseline indicators to remove multicollinearity.

  ### 2. Shift the Dependent Variable
  Instead of predicting the composite `Vulnerability_Score_PCA` using the ingredients that made it, we should either:
    A. Predict one isolated dimension using the others (e.g., can we predict a region's `Lack of Coping Capacity` based purely on its `Climate-Driven Hazard & Exposure`?).
    B. Introduce an external dependent variable if available in your dataset, such as actual displacement figures, GDP impact, or conflict instances.
  ### 3. Interaction Effects (Moderation Analysis)
  Instead of a simple linear model, we can test for interaction effects. For example, we can test if `Lack of Coping Capacity` moderates the effect of `Climate-Driven Hazards`. Statistically, we would add a multiplied term (Hazard $\times$ Coping Capacity) to see if high hazards only cause extreme vulnerability when coping capacity is simultaneously low.